In [ ]:
import os  # added 2026: credentials were scrubbed to environment reads
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
import joblib
import logging
import schedule
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import sys
import json
from config import ALLOWED_ELEMENT_TYPES,ICON_COLOR_MAP
from utils import reformat_scraped_data
from webdriver_manager.chrome import ChromeDriverManager
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# ANSI escape code for green text
GREEN = "\033[92m"
RESET = "\033[0m"

# Configure logging
logging.basicConfig(filename='trading_journal.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Create a console handler that writes to stdout
console_handler = logging.StreamHandler(stream=sys.stdout)
console_handler.setLevel(logging.INFO)

# Define a custom formatter that adds the green color
class CustomFormatter(logging.Formatter):
    def format(self, record):
        log_msg = super().format(record)
        return f"{GREEN}{log_msg}{RESET}"

console_handler.setFormatter(CustomFormatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add the console handler to the root logger
logging.getLogger().addHandler(console_handler)

# Email configuration
SMTP_SERVER = 'smtp.zoho.com'
SMTP_PORT = 587
EMAIL_ADDRESS = os.environ['EMAIL_ADDRESS']
EMAIL_PASSWORD = os.environ['EMAIL_PASSWORD']

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = os.environ['ALERT_RECIPIENT']  # Intended recipient
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_ADDRESS, msg['To'], text)
        server.quit()
        logging.info('Notification Email sent successfully')
    except Exception as e:
        logging.error(f'Failed to send email: {e}')

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus1_tz = pytz.timezone('Etc/GMT-2')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+1
        localized_time = utc_plus1_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df
    
def news_fetch():
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        driver = webdriver.Chrome()
    except:
        print ("AF: No Chrome webdriver installed")
        driver = webdriver.Chrome(ChromeDriverManager().install())

    driver.get("https://www.forexfactory.com/calendar")

    month =  datetime.now().strftime("%B")

    table = driver.find_element(By.CLASS_NAME, "calendar__table")

    data = []
    previous_row_count = 0
    # Scroll down to the end of the page
    while True:
        # Record the current scroll position
        before_scroll = driver.execute_script("return window.pageYOffset;")
        
        # Scroll down a fixed amount
        driver.execute_script("window.scrollTo(0, window.pageYOffset + 500);")
        
        # Wait for a short moment to allow content to load
        time.sleep(2)
        
        # Record the new scroll position
        after_scroll = driver.execute_script("return window.pageYOffset;")
        
        # If the scroll position hasn't changed, we've reached the end of the page
        if before_scroll == after_scroll:
            break

    # Now that we've scrolled to the end, collect the data
    for row in table.find_elements(By.TAG_NAME, "tr"):
        row_data = []
        for element in row.find_elements(By.TAG_NAME, "td"):
            class_name = element.get_attribute('class')
            if class_name in ALLOWED_ELEMENT_TYPES:
                if element.text:
                    row_data.append(element.text)
                elif "calendar__impact" in class_name:
                    impact_elements = element.find_elements(By.TAG_NAME, "span")
                    for impact in impact_elements:
                        impact_class = impact.get_attribute("class")
                        color = ICON_COLOR_MAP[impact_class]
                    if color:
                        row_data.append(color)
                    else:
                        row_data.append("impact")

        if len(row_data):
            data.append(row_data)

    reformat_scraped_data(data,month)
    ds = pd.read_csv(f'{month}_news.csv')
    ds = ds[ds['time'] != 'All Day']
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    convert_time_to_mt5(ds)
    ds['date'] = ds['date'].astype(str).str.strip()
    current_year = datetime.now().year
    ds['date'] = ds['date'] + f' {current_year}'
    ds['date'] = pd.to_datetime(ds['date'], format='%b %d %Y', errors='coerce')
    filename = f"{month}_news.csv"
    ds.to_csv(filename)
    return ds

def get_signal():
    ticker = 'XAUUSD_i'
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 777)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])
        
    objects = joblib.load('U.joblib')
    
    # Shift close price and returns to prevent lookahead bias
    df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
    df['returns_lag_1'] = df['returns'].shift(1)
    df['returns_lag_2'] = df['returns'].shift(2)
    df['returns_lag_3'] = df['returns'].shift(3)
    df['returns_lag_4'] = df['returns'].shift(4)

    # Calculate technical indicators
    df[["lowerBB2", "midBB","upperBB2","bandwidthBB2","percentBB2"]] = ta.bbands(df["Close"], length=21, std=2)
    df.dropna(inplace=True)

    #HMM
    feature_matrix = ['returns', 'returns_lag_1', 'returns_lag_2', 'returns_lag_3', 'returns_lag_4']
    feature_matrix = df[feature_matrix].values
    scaler_5 = objects['scaler_hmm_1']
    feature_matrix_scaled = scaler_5.transform(feature_matrix)
    c_hmm_model = objects['hmm_1']
    df['c_hmm_regime'] = c_hmm_model.predict(feature_matrix_scaled)
    probabilities = c_hmm_model.predict_proba(feature_matrix_scaled)
    df['c_hmm_prob_0'] = probabilities[:, 0]
    df['c_hmm_prob_1'] = probabilities[:, 1]
    df['c_hmm_prob_2'] = probabilities[:, 2]

    #HMM
    feature_matrix = ["bandwidthBB2","percentBB2"]
    feature_matrix = df[feature_matrix].values
    scaler_6 = objects['scaler_hmm_2']
    feature_matrix_scaled = scaler_6.transform(feature_matrix)
    d_hmm_model = objects['hmm_2']
    df['d_hmm_regime'] = d_hmm_model.predict(feature_matrix_scaled)
    probabilities = d_hmm_model.predict_proba(feature_matrix_scaled)
    df['d_hmm_prob_0'] = probabilities[:, 0]
    df['d_hmm_prob_1'] = probabilities[:, 1]
    df['d_hmm_prob_2'] = probabilities[:, 2]

    # Final cleanup
    df.dropna(inplace=True)

    features = ['c_hmm_regime','c_hmm_prob_0', 'c_hmm_prob_1', 'c_hmm_prob_2',
                'd_hmm_regime','d_hmm_prob_0', 'd_hmm_prob_1', 'd_hmm_prob_2']
    X = df[features]
    model = objects['stacking_clf']
    df['pred'] = model.predict(X)
    probs = model.predict_proba(X)
    df['prob_class_-1'] = probs[:, 0]  # Probability of class -1
    df['prob_class_1'] = probs[:, 1]   # Probability of class 1
    prob_1 = df['prob_class_1'].values[-1]
    prob_2 = df['prob_class_-1'].values[-1]
    if prob_1 > prob_2:
        prob = prob_1
    else:
        prob = prob_2

    prob = prob * 100
    prob = round(prob, 2)
    signal = df.pred.values[-1]
    signal_time = df.index[-1]
    signal_prev = df.pred.values[-2]
    upper = df.upper.values[-1]
    lower = df.lower.values[-1]
    
    return signal, signal_time, signal_prev, upper, lower, prob

def calculate_lot_size(entry_price, entry_sl, ticker='XAUUSD_i', risk=2):

    # Get contract size and account balance
    symbol_info = mt5.symbol_info(ticker)

    contract_size = symbol_info.trade_contract_size
    account_balance = mt5.account_info().balance

    # Calculate the stop loss in pips
    sl_pip = round(abs(entry_price - entry_sl) * contract_size, 2)
    
    # Calculate the lot size
    lots = round((account_balance * risk / 100) / sl_pip, 2)
    
    # Ensure the lot size is at least 0.01
    if lots < 0.01:
        lots = 0.01
    
    return lots

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    logging.info(f"Close order result: {result}")
    send_email('Position Closed', f'Position details: {result}')

def execute_trade(signal, qty, upper, lower):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
        sl = lower
        tp = upper
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
        sl = upper
        tp = lower
    else:
        logging.info("No action needed")
        return
    
    logging.info(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    logging.info(f"Trade order result: {result}")
    send_email('Position Opened', f'Position details: {result}')

def modify_open_position(upper, lower):

    # Get all open positions
    positions = mt5.positions_get()

    # Assuming there's only one open position
    position = positions[0]

    # Determine the new SL and TP based on the position type
    if position.type == mt5.ORDER_TYPE_BUY:
        sl = lower
        tp = upper
    elif position.type == mt5.ORDER_TYPE_SELL:
        sl = upper
        tp = lower

    # Create the request to modify the position
    request = {
        "action": mt5.TRADE_ACTION_SLTP,
        "symbol": position.symbol,
        "sl": sl,
        "tp": tp,
        "position": position.ticket,
    }

    # Send the request to modify the position
    result = mt5.order_send(request)
    logging.info(f"Modify position result: {result}")
    send_email('Position Modified', f'Position details: {result}')


def get_mt5_time():
    tz_mt5 = pytz.timezone('Etc/GMT-3')  # Use the correct timezone
    now = datetime.now(tz_mt5)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    

def get_next_bar_time(interval):
    now = get_mt5_time()
    now = datetime.strptime(now, '%Y-%m-%d %H:%M:%S')
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")

# Schedule the function to run every day at midnight in the specified timezone
def schedule_job():
    timezone = pytz.timezone('Etc/GMT-3')
    now = datetime.now(timezone)
    schedule_time = now.replace(hour=0, minute=0, second=0, microsecond=0)

    # If it's already past midnight, schedule for the next day
    if now > schedule_time:
        schedule_time += timedelta(days=1)

    schedule.every().day.at(schedule_time.strftime("%H:%M")).do(news_fetch)

def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()
    
    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            logging.info(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            logging.info(f"Next check time: {next_check_time}")

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 24"
            news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce')
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            #print(f"Current date: {current_date}")
            logging.info(f"News events for today:\n{news_df_today[['date', 'time', 'currency','impact','event']]}")
            logging.info(f"--------------------------------------")
                             
            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                news_time = news_time.replace(year=MT5.year, month=MT5.month, day=MT5.day)  # Ensure the correct date
                while news_time - timedelta(minutes=29) <= MT5 < news_time + timedelta(minutes=15):
                    # Fetch open position
                    open_position = get_open_position()
                    
                    if open_position:
                        close_position(open_position)
                        logging.info("Closed open position due to upcoming news event.")
                    
                    if not message_logged:  # Log only if the message hasn't been logged yet
                        logging.info(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                        message_logged = True  # Set the flag to True after logging the message
                    
                    MT5 = get_mt5_time()
                    MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                    time.sleep(1)
                

            # Fetch current signal and its time
            current_signal, signal_time, signal_prev, upper, lower, prob = get_signal()
            logging.info(f"New signal checked: {current_signal}, Signal time: {signal_time}, The probability: {prob} %")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                logging.info("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            ticker='XAUUSD_i'
            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    logging.info(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1, 0, 0)
                        # if current_signal == 1:
                        #     entry = mt5.symbol_info_tick(ticker).ask
                        #     sl = lower
                        #     lot = calculate_lot_size(entry, sl)
                        #     if entry < upper:
                        #         execute_trade(current_signal, 1, 0, 0)
                        #     else:
                        #         logging.info("Out of BB, not allowed.")
                        # elif current_signal == -1:
                        #     entry = mt5.symbol_info_tick(ticker).bid
                        #     sl = upper
                        #     lot = calculate_lot_size(entry, sl)
                        #     if entry > lower:
                        #         execute_trade(current_signal, 1, 0, 0)
                        #     else:
                        #         logging.info("Out of BB, not allowed.")
                    else:
                        logging.info("Failed to close the position. Not executing new trade.")
                else:
                    logging.info(f"Position exists and it is the same.")
                    # modify_open_position(upper, lower)
            else:
                execute_trade(current_signal, 1, 0, 0)
                #if current_signal != signal_prev:
                    # if current_signal == 1:
                    #     entry = mt5.symbol_info_tick(ticker).ask
                    #     sl = lower
                    #     lot = calculate_lot_size(entry, sl)
                    #     if entry < upper:
                    #         execute_trade(current_signal, lot, upper, lower)
                    #     else:
                    #         logging.info("Out of BB, not allowed.")
                    # elif current_signal == -1:
                    #     entry = mt5.symbol_info_tick(ticker).bid
                    #     sl = upper
                    #     lot = calculate_lot_size(entry, sl)
                    #     if entry > lower:
                    #         execute_trade(current_signal, lot, upper, lower)
                    #     else:
                    #         logging.info("Out of BB, not allowed.")
                #else:
                    #print("previous order hit SL or TP and this one is in the same direction, so we passed.")

        except Exception as e:
            logging.info(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

# Create a stop event
stop_event = Event()

#get the red news
news_fetch()

# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

In [ ]:
try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    news_df = pd.read_csv(file_name)

    # Start the trading loop
    check_and_trade(stop_event, news_df)

except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")